# <center>Trabajo 1 - Análisis espacial de la relación entre la tasa de natalidad y la tasa de alfabetización en la Provincia del Maipo: Comparación censal 2017–2024</center>
### <center><font color="grey">*Helen Aguayo Silva*</font></center>
### <center><font color="grey">*Inteligencia en Datos Geoespaciales*</font></center>
### <center><font color="grey">*Primer semestre 2026*</font></center>

In [2]:
# Librerías para mapas 2017
! pip install pyreadr
!conda install -y pyarrow
%pip install sqlalchemy psycopg2-binary
from sqlalchemy import create_engine
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import mapclassify 
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
# Librerías para mapas 2024
import numpy as np
import matplotlib.pyplot as plt
import pyreadr

Jupyter detected...
3 channel Terms of Service accepted
Channels:
 - defaults
Platform: win-64
Solving environment: ...working... done

# All requested packages already installed.



"powershell" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


  Using cached sqlalchemy-2.0.51-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached psycopg2_binary-2.9.12-cp313-cp313-win_amd64.whl.metadata (5.1 kB)
  Using cached greenlet-3.5.4-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
Using cached sqlalchemy-2.0.51-cp313-cp313-win_amd64.whl (2.1 MB)
Using cached psycopg2_binary-2.9.12-cp313-cp313-win_amd64.whl (2.8 MB)
Using cached greenlet-3.5.4-cp313-cp313-win_amd64.whl (247 kB)

   ------------- -------------------------- 1/3 [greenlet]
   ------------- -------------------------- 1/3 [greenlet]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   ------------------

ModuleNotFoundError: No module named 'geopandas'

In [ ]:
DB_HOST     = 'localhost'
DB_PORT     = 5432
DB_NAME     = 'censo_rm'
DB_USER     = 'postgres'
DB_PASSWORD = 'postgres'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

In [ ]:
sql_indicadores = '''
WITH agg AS (
SELECT 
c.nom_comuna, 
z.geocodigo::DOUBLE PRECISION AS geocodigo,

ROUND((COUNT(*) FILTER (WHERE p.p15a = 1 AND p.p09 >= 15) * 100.0) / NULLIF(COUNT(*) FILTER (WHERE p.p09 >= 15), 0), 2) AS tasa_alfabetizacion,

ROUND((COUNT(*) FILTER (WHERE p.p09 <= 4) * 100.0) / NULLIF(COUNT(*) FILTER (WHERE p.p08 = 2 AND p.p09 BETWEEN 15 AND 49), 0), 2) AS tasa_natalidad

FROM public.personas AS p
JOIN public.hogares AS h 
ON p.hogar_ref_id = h.hogar_ref_id
JOIN public.viviendas AS v 
ON h.vivienda_ref_id = v.vivienda_ref_id
JOIN public.zonas AS z 
ON v.zonaloc_ref_id = z.zonaloc_ref_id
JOIN public.comunas AS c 
ON z.codigo_comuna = c.codigo_comuna
JOIN public.provincias AS pr 
ON pr.provincia_ref_id = c.provincia_ref_id
WHERE pr.nom_provincia = 'MAIPO'
GROUP BY c.nom_comuna, z.geocodigo
)
SELECT a.*, shp.geom
FROM agg AS a
JOIN dpa.zonas_censales_rm AS shp 
ON shp.geocodigo = a.geocodigo;
'''

In [ ]:
gdf = gpd.read_postgis(sql_indicadores, engine, geom_col='geom')

In [ ]:
gdf

In [ ]:
gdf[['tasa_natalidad','tasa_alfabetizacion']].describe().round(2)

In [ ]:
# 1. Filtrar comunas de Maipo
comunas_maipo = ['SAN BERNARDO', 'BUIN', 'PAINE', 'CALERA DE TANGO']
gdf_maipo = gdf[gdf['nom_comuna'].str.upper().isin(comunas_maipo)].copy()

# 2. LIMPIEZA DE HUECOS: Eliminar líneas/geometrías internas y reparar huecos
gdf_maipo = gdf_maipo[
    gdf_maipo['geom'].geom_type.isin(['Polygon', 'MultiPolygon'])
]

# 3. Crear figura
fig, ax = plt.subplots(figsize=(10, 10))

# 4. Asignar colores por comuna
paleta = sns.color_palette('Set2', n_colors=4)
colores_comuna = dict(zip(comunas_maipo, paleta))
gdf_maipo['color_comuna'] = (
    gdf_maipo['nom_comuna'].str.upper().map(colores_comuna)
)

# 5. Dibujar Zonas Censales
gdf_maipo.plot(
    ax=ax, color=gdf_maipo['color_comuna'], edgecolor='white', linewidth=0.2
)

# 6. Disolver comunas y dibujar SOLAMENTE el perímetro exterior (sin dibujar hoyos/rings internos)
comunas = gdf_maipo.dissolve(by='nom_comuna', as_index=False)

# Extraer y dibujar solo el contorno exterior envolvente de cada comuna
for _, row in comunas.iterrows():
    geom = row['geom']
    # Si es MultiPolygon tomamos los exteriores de cada polígono
    poligonos = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
    for poly in poligonos:
        x_ext, y_ext = poly.exterior.xy
        ax.plot(x_ext, y_ext, color='#1a252f', linewidth=1.5)

# 7. Desplazamientos directos para nombres
offsets = {
    'CALERA DE TANGO': (-8000, 3000),
    'SAN BERNARDO': (6000, -2000),
}

if gdf_maipo.crs and gdf_maipo.crs.is_geographic:
    offsets = {'CALERA DE TANGO': (-0.08, 0.03), 'SAN BERNARDO': (0.06, -0.02)}

# 8. Texto con halo blanco
for _, row in comunas.iterrows():
    nombre = row['nom_comuna'].upper()
    c = row['geom'].centroid

    dx, dy = offsets.get(nombre, (0, 0))
    x, y = c.x + dx, c.y + dy

    ax.text(
        x,
        y,
        row['nom_comuna'].title(),
        ha='center',
        va='center',
        weight='bold',
        fontsize=11,
    ).set_path_effects([pe.withStroke(linewidth=3.5, foreground='white')])

# 9. Título y formato (sin simbología)
ax.set_title(
    'Provincia de Maipo — Zonas Censales por Comuna', weight='bold', pad=12
)
ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 1. Filtrar los datos para la Provincia de Maipo
comunas_maipo = ['SAN BERNARDO', 'BUIN', 'PAINE', 'CALERA DE TANGO']
gdf_maipo = gdf[gdf['nom_comuna'].str.upper().isin(comunas_maipo)].copy()

# 2. Filtro anti-líneas / geometrías no válidas
gdf_maipo = gdf_maipo[
    gdf_maipo['geom'].geom_type.isin(['Polygon', 'MultiPolygon'])
]

# 3. Disolver comunas para obtener límites exteriores y centroides
comunas = gdf_maipo.dissolve(by='nom_comuna', as_index=False)

# Desplazamientos personalizados para evitar choque de etiquetas
offsets = {
    'CALERA DE TANGO': (-8000, 3000),
    'SAN BERNARDO': (6000, -2000),
}
if gdf_maipo.crs and gdf_maipo.crs.is_geographic:
    offsets = {'CALERA DE TANGO': (-0.08, 0.03), 'SAN BERNARDO': (0.06, -0.02)}


# 4. Función de mapeo univariado con nuestro estilo limpio
def plot_univariado_estilo(ax, gdf_z, gdf_c, col, cmap, titulo, cbar_label):
    # A. Mapa de coropletas por zona censal
    gdf_z.plot(
        column=col,
        ax=ax,
        cmap=cmap,
        linewidth=0.2,
        edgecolor='white',
        legend=True,
        legend_kwds={'label': cbar_label, 'shrink': 0.55, 'pad': 0.02},
    )

    # B. Dibujar ÚNICAMENTE el borde exterior de las comunas (estilo perimetral limpio)
    for _, row in gdf_c.iterrows():
        geom = row['geom']
        poligonos = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
        for poly in poligonos:
            x_ext, y_ext = poly.exterior.xy
            ax.plot(x_ext, y_ext, color='#1a252f', linewidth=1.4)

    # C. Nombres de las comunas con halo blanco
    for _, row in gdf_c.iterrows():
        nombre = row['nom_comuna'].upper()
        c = row['geom'].centroid

        dx, dy = offsets.get(nombre, (0, 0))
        x, y = c.x + dx, c.y + dy

        ax.text(
            x,
            y,
            row['nom_comuna'].title(),
            ha='center',
            va='center',
            weight='bold',
            fontsize=10,
        ).set_path_effects([pe.withStroke(linewidth=3, foreground='white')])

    ax.set_title(titulo, fontsize=12, fontweight='bold', pad=12)
    ax.axis('off')


# 5. Crear la figura con 2 subplots lado a lado
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.patch.set_facecolor('#ffffff')

# Mapa 1: Tasa de Alfabetización
plot_univariado_estilo(
    axes[0],
    gdf_maipo,
    comunas,
    col='tasa_alfabetizacion',
    cmap='Greens',
    titulo='Tasa de Alfabetización por Zona Censal — Prov. Maipo',
    cbar_label='Tasa de Alfabetización',
)

# Mapa 2: Tasa de Natalidad
plot_univariado_estilo(
    axes[1],
    gdf_maipo,
    comunas,
    col='tasa_natalidad',
    cmap='Purples',
    titulo='Tasa de Natalidad por Zona Censal — Prov. Maipo',
    cbar_label='Tasa de Natalidad',
)

plt.tight_layout(pad=2)
plt.show()

In [ ]:
# 1. Preparar datos de Maipo y clasificar por terciles (3x3)
comunas_maipo = ['SAN BERNARDO', 'BUIN', 'PAINE', 'CALERA DE TANGO']
gdf_maipo = gdf[gdf['nom_comuna'].str.upper().isin(comunas_maipo)].copy()
gdf_maipo = gdf_maipo[
    gdf_maipo['geom'].geom_type.isin(['Polygon', 'MultiPolygon'])
]

# Terciles para las variables
gdf_maipo['var1_cat'] = pd.qcut(
    gdf_maipo['tasa_natalidad_1k'], q=3, labels=[1, 2, 3]
).astype(int)
gdf_maipo['var2_cat'] = pd.qcut(
    gdf_maipo['tasa_alfabetizacion'], q=3, labels=[1, 2, 3]
).astype(int)

gdf_maipo['bivariate_code'] = gdf_maipo.apply(
    lambda r: f"{r['var1_cat']}{r['var2_cat']}", axis=1
)

# Paleta de colores exactamente como la de tu imagen (Rosa/Púrpura vs Verde)
bivariate_colors = {
    '11': '#e8e8e8',
    '21': '#a2c896',
    '31': '#4fa060',  # Fila baja (Gris a Verde)
    '12': '#d1b0d5',
    '22': '#9292b8',
    '32': '#427863',  # Fila media (Lila a Verde Oscuro)
    '13': '#98669f',
    '23': '#675193',  # Fila alta (Morado a Verde Muy Oscuro/Negro)
    '33': '#293e36',
}

gdf_maipo['color_bivariado'] = gdf_maipo['bivariate_code'].map(bivariate_colors)

# ---------------------------------------------------------
# 2. Configurar la Figura y el Lienzo (Estilo Póster Beige)
# ---------------------------------------------------------
bg_color = '#f5f3e9'  # Color de fondo beige suave
ocean_color = '#aed2e6'  # Azul claro para el recuadro del mapa

fig = plt.figure(figsize=(14, 10), facecolor=bg_color)

# Crear ejes para el mapa (rectángulo azul a la izquierda)
ax_map = fig.add_axes([0.15, 0.05, 0.50, 0.82], facecolor=ocean_color)

# 3. Dibujar el Mapa
gdf_maipo.plot(
    ax=ax_map,
    color=gdf_maipo['color_bivariado'],
    edgecolor='white',
    linewidth=0.2,
)

# Dibujar únicamente contornos exteriores de comunas
comunas = gdf_maipo.dissolve(by='nom_comuna', as_index=False)
for _, row in comunas.iterrows():
    geom = row['geom']
    poligonos = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
    for poly in poligonos:
        x_ext, y_ext = poly.exterior.xy
        ax_map.plot(x_ext, y_ext, color='#111111', linewidth=1.6)

# Etiquetas de comunas
offsets = {
    'CALERA DE TANGO': (-8000, 3000),
    'SAN BERNARDO': (6000, -2000),
}
if gdf_maipo.crs and gdf_maipo.crs.is_geographic:
    offsets = {'CALERA DE TANGO': (-0.08, 0.03), 'SAN BERNARDO': (0.06, -0.02)}

for _, row in comunas.iterrows():
    nombre = row['nom_comuna'].upper()
    c = row['geom'].centroid

    dx, dy = offsets.get(nombre, (0, 0))
    x, y = c.x + dx, c.y + dy

    ax_map.text(
        x,
        y,
        nombre,
        ha='center',
        va='center',
        weight='bold',
        fontsize=10,
        color='black',
        bbox=dict(
            boxstyle='round,pad=0.3', fc='white', ec='none', alpha=0.85
        ),  # Estilo de etiqueta similar
    )

ax_map.axis('off')

# ---------------------------------------------------------
# 4. Encabezado y Títulos Superiores
# ---------------------------------------------------------
# Título Principal Centrado arriba
fig.text(
    0.40,
    0.95,
    'MAPA BIVARIADO',
    fontsize=20,
    fontweight='bold',
    ha='center',
    color='#1c2333',
)
fig.text(
    0.40,
    0.92,
    '% Natalidad  ×  % Alfabetización',
    fontsize=12,
    style='italic',
    ha='center',
    color='#4a5568',
)
fig.text(
    0.40,
    0.89,
    'Provincia de Maipo  .  Zonas Censales  .  Censo 2024',
    fontsize=9,
    ha='center',
    color='#718096',
)

# Línea divisoria horizontal superior
line_ax = fig.add_axes([0.05, 0.88, 0.70, 0.002])
line_ax.patch.set_facecolor('#2d3748')
line_ax.axis('off')

# ---------------------------------------------------------
# 5. Leyenda Matriz 3x3 en el lado derecho
# ---------------------------------------------------------
ax_leg = fig.add_axes([0.72, 0.20, 0.22, 0.22], facecolor='none')

for i in range(1, 4):  # Var 1: Natalidad (X)
    for j in range(1, 4):  # Var 2: Alfabetización (Y)
        code = f'{i}{j}'
        color = bivariate_colors[code]
        rect = mpatches.Rectangle(
            (i - 1, j - 1), 1, 1, facecolor=color, edgecolor='white', lw=1.5
        )
        ax_leg.add_patch(rect)

ax_leg.set_xlim(0, 3)
ax_leg.set_ylim(0, 3)
ax_leg.set_title('Leyenda', fontsize=10, fontweight='bold', pad=12)

# Ejes de la leyenda
ax_leg.set_xlabel('% Natalidad →', fontsize=8, fontweight='bold', labelpad=15)
ax_leg.set_ylabel(
    '% Alfabetización →', fontsize=8, fontweight='bold', labelpad=15
)

# Quitar marcas de los ejes de la leyenda
ax_leg.set_xticks([])
ax_leg.set_yticks([])
for spine in ax_leg.spines.values():
    spine.set_color('#cbd5e0')

plt.show()

In [ ]:
RUTA_CENSO_ZONAL = "...\idg_1S_2026\data\PARQUET\Cartografía_censo2024_R13_Zonal.parquet"